In [4]:
import seaborn as sns
import pandas as pd
sns.set(font_scale=1.5)
import matplotlib.pyplot as plt
import numpy as np
from sklearn import linear_model

import plotly.graph_objects as go
import plotly.express as px

from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.utils import shuffle
from sklearn.linear_model import Ridge
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import GridSearchCV

In [5]:
from sklearn.datasets import load_iris

In [6]:
iris = load_iris()

In [7]:
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)

In [14]:
len(df)

150

In [15]:
import random
random.seed(42)

islands = ['blue', 'green', 'red', 'yellow']
df['island'] = [random.choice(islands) for _ in range(len(df))]

df['island'].value_counts()

island
green     42
blue      40
red       37
yellow    31
Name: count, dtype: int64

In [16]:
df['target'] = iris.target

In [17]:
df['species'] = df['target'].map({i: name for i, name in enumerate(iris.target_names)})

In [18]:
df.columns

Index(['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)',
       'petal width (cm)', 'island', 'target', 'species'],
      dtype='str')

In [19]:
numeric_features = ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)','petal width (cm)']

In [20]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_features = ['island']

In [21]:
all_indices = range(0, len(df))
all_indices = shuffle(all_indices)
training_indices, dev_indices = np.split(all_indices, [320])
training_data = df.iloc[training_indices]
dev_data = df.iloc[dev_indices]

In [23]:
training_data.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),island,target,species
29,4.7,3.2,1.6,0.2,blue,0,setosa
22,4.6,3.6,1.0,0.2,yellow,0,setosa
97,6.2,2.9,4.3,1.3,yellow,1,versicolor
104,6.5,3.0,5.8,2.2,blue,2,virginica
133,6.3,2.8,5.1,1.5,yellow,2,virginica


In [24]:
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features),
    ('num', StandardScaler(), numeric_features)
])

In [25]:
scaled_lasso_model = Pipeline([
    ('preprocess', preprocessor),
    ('transform', PolynomialFeatures(degree=3, include_bias=False)),
    ('lasso', Lasso())
])


In [26]:
parameters_to_try = {'lasso__alpha': 10**np.linspace(-4, 4, 100)}

In [27]:
model_finder = GridSearchCV(estimator = scaled_lasso_model,
                               param_grid = parameters_to_try,
                               scoring = "neg_mean_squared_error",
                               cv=[[training_indices, dev_indices]])

In [28]:
model_finder.fit(df[numeric_features + categorical_features], df["target"])

/home/fazuskazoo/anaconda3/envs/ucb/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.079e+00, tolerance: 1.000e-02
  model = cd_fast.enet_coordinate_descent(
/home/fazuskazoo/anaconda3/envs/ucb/lib/python3.14/site-packages/sklearn/model_selection/_validation.py:927: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/fazuskazoo/anaconda3/envs/ucb/lib/python3.14/site-packages/sklearn/model_selection/_validation.py", line 916, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "/home/fazuskazoo/anaconda3/envs/ucb/lib/python3.14/site-packages/sklearn/metrics/_scorer.py", line 317, in __call__
    return self._score(partial(_cached_cal

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...o', Lasso())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.",{'lasso__alpha': array([1.0000...00000000e+04])}
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.","[[array([ 29, ...19, 89, 108]), array([], dtype=int64)]]"
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and param

In [29]:
model_finder.best_estimator_.named_steps['lasso'].coef_

array([ 1.04081246e-02,  9.44090354e-03, -0.00000000e+00,  1.23003456e-01,
       -2.34172690e-01, -6.85904615e-02,  9.06002672e-01,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00, -7.99928747e-02,  4.03620153e-02,
        0.00000000e+00,  7.33727214e-02,  2.33397023e-02,  0.00000000e+00,
       -4.75310007e-02, -1.03084655e-01, -3.06050652e-02,  1.21946391e-01,
       -2.56838556e-04,  8.17094126e-02, -3.73188075e-02, -6.57826442e-02,
        1.24438234e-01, -0.00000000e+00,  1.66175684e-01,  2.98443937e-02,
       -1.47244108e-01, -6.10196855e-02, -3.43612096e-01,  2.97947736e-01,
        2.90161871e-01, -1.58464202e-01,  2.29378613e-01,  1.18770939e-05,
        0.00000000e+00,  0.00000000e+00, -5.84558824e-02,  3.37827608e-02,
        0.00000000e+00,  6.34046205e-02,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00, -

In [30]:
best_model = model_finder.best_estimator_

In [31]:
#lasso_weights = pd.DataFrame([best_model.named_steps["lasso"].coef_],
#                             columns = best_model.named_steps["transform"].get_feature_names_out())
lasso_weights = pd.DataFrame([model_finder.best_estimator_.named_steps["lasso"].coef_],
                             columns=model_finder.best_estimator_.named_steps["transform"].get_feature_names_out())


In [32]:
lasso_weights

,x0,x1,x2,x3,x4,x5,x6,x0^2,x0 x1,x0 x2,...,x4^3,x4^2 x5,x4^2 x6,x4 x5^2,x4 x5 x6,x4 x6^2,x5^3,x5^2 x6,x5 x6^2,x6^3
0,0.010408,0.009441,-0.0,0.123003,-0.234173,-0.06859,0.906003,0.0,0.0,0.0,...,-0.013678,-0.064031,-0.12764,-0.169484,0.014535,0.0,0.189287,0.0,-0.017717,-0.169725


In [43]:
# Get feature names directly from the preprocessor step
ohe_features = model_finder.best_estimator_ \
                            .named_steps['preprocess'] \
                            .named_transformers_['cat'] \
                            .get_feature_names_out(categorical_features)

# Combine OHE names with numeric names
# Note: OHE features come first because 'cat' was listed first in ColumnTransformer
all_feature_names = list(ohe_features) + numeric_features

# Now get polynomial feature names using the correct input names
feature_names = model_finder.best_estimator_ \
                            .named_steps['transform'] \
                            .get_feature_names_out(all_feature_names)

# Rebuild the dataframe
coefficients = model_finder.best_estimator_ \
                           .named_steps['lasso'].coef_

coef_df = pd.DataFrame({"features": feature_names, "coefficients": coefficients})
sortdf = coef_df.sort_values(by="coefficients", key=abs, ascending=False)
print(sortdf[sortdf['coefficients'] != 0])

                                              features  coefficients
6                                     petal width (cm)      0.906003
91   island_yellow sepal length (cm) petal length (cm)      0.768558
95     island_yellow sepal width (cm) petal width (cm)     -0.734497
94    island_yellow sepal width (cm) petal length (cm)      0.581866
89                   island_yellow sepal length (cm)^2     -0.517211
..                                                 ...           ...
56     island_green sepal length (cm) petal width (cm)     -0.004270
57                     island_green sepal width (cm)^2     -0.004124
102               sepal length (cm)^2 petal width (cm)     -0.001798
20                                     island_yellow^2     -0.000257
35                                      island_green^3      0.000012

[83 rows x 2 columns]


In [39]:
feature_counts

,feature,non_zero_terms
0,sepal length (cm),0
1,sepal width (cm),0
2,petal length (cm),0
3,petal width (cm),0
